In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path("For_Thesis_finetune")

FILES = sorted(ROOT.glob("finetune_datasets/*.json")) + sorted(ROOT.glob("candidate_gen_*.json"))


def count(path):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return len(data)
rows = [
    {
        "file": p.name,
        "folder": p.parent.name,
        "examples": count(p),
        "size_MB": round(p.stat().st_size / 1e6, 2),
    }
    for p in FILES
]

df = pd.DataFrame(rows)

In [2]:
def family(name):
    return name.replace("_train.json", "").replace("_valid.json", "")
def split(name):
    return "valid" if "_valid" in name else "train"
summary = (
    df.assign(dataset=df["file"].map(family), split=df["file"].map(split))
    .pivot_table(index="dataset", columns="split", values="examples", aggfunc="sum", fill_value=0)
)
summary["total"] = summary.sum(axis=1)
summary["valid_%"] = (100 * summary["valid"] / summary["total"]).round(1)
summary

split,train,valid,total,valid_%
dataset,,,,
candidate_gen,13406,1589,14995,10.6
ft_slm_context,4031,519,4550,11.4
ft_slm_limited,2975,276,3251,8.5
